<img src="images/banner.png" style="width: 100%;">

# <span style="color:darkblue">**The Healthcare Paradox**: Predicting Hospital Level Gaps Across Philippine LGUs Using Socioeconomic and Demographic Modeling</span>

## <span style="color:darkblue">1 Executive Summary</span>

The Philippines operates a tiered hospital system regulated by the Department of Health (DOH), in which Level 1 facilities provide basic inpatient care, Level 2 facilities support general surgery and internal medicine, and Level 3 facilities offer full specialty care with intensive care units. Despite this structured framework, a persistent and well-documented gap exists between *where hospitals are located* and *where they are most needed*. Based on the data we have gathered, more than 78% of the country's 1,631 local government units (LGUs) have no hospital of any level, and the highest-tier hospitals are concentrated in wealthier, more urbanized areas. This leaves the poorest and fastest-growing municipalities systematically underserved. This project addresses the question: ***Can we predict what health facility category by service capability (Level 1, Level 2, Level 3) an LGU has based on its socioeconomic profile? And if so, which specific cities/municipalities are being left behind in the Philippines?***

Data were collected from five Philippine government sources—the DOH National Health Facilities Registry (NHFR, ~44,000 records), the PSA Census (2020 and 2024 populations across 18 regional sheets), PSA poverty incidence files (2018, 2021, 2023), and PSA registered live births (2023)—supplemented by infrastructure amenity counts scraped from OpenStreetMap (OSM) via the Overpass API. After deduplication, imputation, feature engineering, standardization, and PCA-based dimensionality reduction, a final dataset of 1,631 LGUs with 17 features was constructed and stored in an SQLite database. Five supervised machine learning models—k-Nearest Neighbours, Logistic Regression (L1 and L2), Random Forest, and Gradient Boosted Trees (GBM)—were trained to predict an ordinal Hospital Level (0–3) per LGU. The best model, GBM, achieved a macro F1 of 0.6959 and 90% weighted accuracy. The key findings are: (1) birth demand and population growth are the strongest predictors of which hospital level an LGU needs; (2) poverty incidence is a strong feature in the model, but it operates in the inequitable direction, where *higher poverty incidence level* predicts *lower* tier. This confirms that the current system allocates hospitals based on economic capacity rather than health need; and (3) the model identifies LGUs whose socioeconomic profiles predict a higher hospital tier than they currently hold. The main recommendation is that DOH infrastructure investment should be guided by the model's Expected Tier score—a priority metric derived from the GBM's class probabilities—with explicit preference for high-poverty, high-birth-demand LGUs where the gap between predicted and actual tier is largest.

## <span style="color:darkblue">2 Introduction</span>

Access to hospital care is one of the most consequential determinants of health outcomes, yet in the Philippines it remains significantly unequal across geography and socioeconomic class. The DOH classifies hospitals into three service capability levels: Level 1 (primary, basic inpatient and minor surgery), Level 2 (secondary, general surgery and internal medicine departments), and Level 3 (tertiary, with specialty services, intensive care units, and residency training). The distribution of these facilities across the country's 1,631 municipalities and cities does not reflect the distribution of health need; it reflects the distribution of historical investment, urbanisation, and economic activity.

The Philippine Universal Health Care Act of 2019 (Republic Act 11223) mandates equitable access to quality health services across all LGUs. However, planning and resource allocation still lack a systematic, data-driven method for identifying which LGUs need hospital upgrades most urgently. Health system managers currently rely on population-based ratios (e.g., 1 hospital per 500,000 people) that fail to account for geographic heterogeneity in poverty, birth rates, and infrastructure development.

The objective of this project is to build a data mining and wrangling along with machine learning pipeline that predicts the hospital level an LGU *should* have based on its measurable socioeconomic and demographic characteristics, then uses the gap between predicted and actual tier to generate a ranked list of structurally underserved LGUs. 

The scope of this project covers all 1,631 municipalities and cities in the Philippines with complete or imputable data. The analysis is cross-sectional, using the most recent available data for each feature (2020–2024 depending on source). Limitations include: (a) the DOH NHFR "Service Capability" field is blank for some older hospital registrations, causing those facilities to be excluded from level counts; (b) OSM data quality varies by region, with some rural LGUs having sparse amenity records; and (c) the poverty incidence data has ~5.6% missing values, addressed by regional median imputation.

## <span style="color:darkblue">3 Business Value</span>

Hospital infrastructure planning in the Philippines currently lacks a systematic, data-driven framework for prioritising LGUs in need of facility upgrades. This project creates measurable value for the following stakeholders.

**Department of Health — Health Facilities and Services Regulatory Bureau (HFSRB).** The HFSRB is responsible for the licensing and upgrading of hospital facilities nationwide. The model's Expected Tier score and tier-gap ranking provide a decision-support tool for the DOH's annual Hospital Development Plan. This would enable budget officers to rank upgrade candidates using an objective, reproducible metric rather than ad hoc lobbying or legacy prioritisation.

**Philippine Health Insurance Corporation (PhilHealth).** PhilHealth's case rates and provider accreditation policies are linked to hospital DOH level. Identifying LGUs where the population profile warrants a Level 2 or Level 3 hospital, but none exists, shows gaps in the insurance network where beneficiaries are forced to travel to urban hospitals, incurring higher out-of-pocket costs. The model's output can directly inform PhilHealth's network expansion planning.

**Local Government Units and Sangguniang Panlalawigan.** Provincial health officers and local legislators can use the ranked underserved-LGU list to make evidence-based budget requests to the Mandanas-Garcia devolution agenda, or the Philippine Supreme Court ruling that expanded the tax base for computing LGUs' internal revenue shares). LGUs that appear in the top tier-gap decile can cite the model's Expected Tier score in budget memoranda submitted to the Department of Budget and Management (DBM).

**National Economic and Development Authority (NEDA).** NEDA's Regional Development Plans include health infrastructure targets. The project's equity finding—that poverty incidence suppresses hospital tier independently of birth demand—provides quantitative evidence for a structural argument: the market has systematically failed to allocate hospitals to the most vulnerable LGUs. 

**Academic and Policy Research Community.** The publicly available pipeline (GitHub repository, SQLite database, and cleaned datasets) provides a reproducible baseline for future studies on health facility equity in Southeast Asia.

The risks addressed include misallocation of healthcare capital, inequitable access, and inefficiency in DOH licensing workflows. The most direct measurable impact is a reduction in facility-to-need mismatch, quantified by the tier-gap metric produced by this model.

## <span style="color:darkblue">4 Data</span>

### 4.1 Data Sources and Extraction

Five government datasets were used, supplemented by one web-scraped infrastructure source. Table 1 summarises each source.

<a id="table1"></a>
<p style="font-size:12px;font-style:default;"><b>Table 1. Data sources, record counts, and extraction method</b></p>

| Source | Description | Records | Access Method |
|--------|-------------|---------|---------------|
| DOH National Health Facilities Registry (NHFR) | All licensed health facilities; each sub-service as a separate row | ~44,000 raw rows | Downloaded from DOH Open Data portal (XLSX) |
| PSA Census 2020 + 2024 | Municipal/city population across 18 regional worksheets | 1,631 LGUs | Downloaded from PSA (XLSX, multi-sheet) |
| PSA Poverty Incidence (municipal) | Municipal-level poverty incidence 2018, 2021, 2023 | ~1,613 municipalities | Downloaded from PSA (XLSX) |
| PSA Poverty Incidence (NCR HUC) | NCR highly urbanised city poverty incidence | 16 NCR HUCs | Downloaded from PSA (XLSX, tab1a) |
| PSA Registered Live Births 2023 | Births by place of occurrence and mother's residence | 1,631 LGUs | Downloaded from PSA (XLSX) |
| OpenStreetMap / Overpass API | Amenity counts per LGU bounding box (ATM, bank, pharmacy, school, etc.) | 18 amenity types × 1,631 LGUs | Scraped via Overpass API (`00_scrape_data.py`) |

The OSM scraping script (`00_scrape_data.py`) submits Overpass QL queries against the public Overpass API endpoint, filtering by LGU administrative boundaries. A crawl delay is enforced between requests to comply with Overpass API usage guidelines. Results are parsed into amenity count vectors per LGU.

The DOH NHFR required cleaning at ingestion because each licensed sub-service within a hospital (drug testing unit, dialysis bay, ambulatory surgical unit) receives its own row. The deduplication strategy is described in Section 5.

### 4.2 Data Description

After cleaning and merging, the final dataset contains **1,631 LGU-level observations** and **30 features** (12 socioeconomic + 18 infrastructural), plus 3 target variables. The dataset is stored in a SQLite database (`hospital_data.db`) with 7 normalised sub-tables linked by a surrogate `lgu_id` primary key, plus one denormalised `lgu_merged` table that serves as the single source of truth for downstream modelling.

<a id="table2"></a>
<p style="font-size:12px;font-style:default;"><b>Table 2. Feature groups, variable names, and data types</b></p>

| Group | Features | Type | Description |
|-------|---------|------|-------------|
| **Population demand** | `population_2020`, `population_2024`, `pop_growth_rate_pct` | Numeric | PSA Census; growth rate computed as `(pop_2024 − pop_2020) / pop_2020 × 100` |
| **Poverty (barrier)** | `poverty_incidence_2018_pct`, `_2021_pct`, `_2023_pct` | Numeric | % families below poverty line (PSA, three reference years) |
| **Birth demand** | `births_occurrence_both/male/female`, `births_residence_both/male/female` | Numeric | PSA live births by occurrence location and mother's usual residence |
| **OSM infrastructure** | `atm`, `bank`, `bar`, `bus_station`, `cafe`, `community_centre`, `fast_food`, `fuel`, `parking`, `pharmacy`, `place_of_worship`, `police`, `post_office`, `restaurant`, `school`, `shelter`, `toilets`, `townhall` | Numeric (count) | Amenity counts per LGU from OpenStreetMap via Overpass API |
| **Target** | `hospital_count_level1/2/3` | Numeric (integer) | DOH Level 1/2/3 hospitals in LGU |

### 4.3 Data Quality Issues

Two categories of missing values were identified in the merged dataset. Poverty columns have approximately **5.6% missing** values, concentrated in newly created LGUs not yet covered in PSA poverty surveys. Birth columns have approximately **10.8% missing** values, concentrated in LGUs with populations below the PSA reporting threshold. Both are addressed by regional median imputation (see Section 5.3).

A structural limitation of the NHFR is that the `Service Capability` field is blank for some older hospital registrations. These facilities receive `doh_level = 0` and are excluded from level counts even if they operate at Level 2 capacity. They are included in `total_hospitals` for reference but not in the tier targets.

OSM data coverage is uneven: urbanised cities have dense amenity records, while rural LGUs may have sparse or absent coverage. Infrastructure features with more than 75% zeros across all LGUs were dropped during feature selection.

## <span style="color:darkblue">5 Methodology</span>

The pipeline spans four Python scripts (`00`–`03`) and one modelling notebook (`04`) with two different versions, structured so that all data extraction, wrangling, storage, and dimensionality reduction occur in standalone Python modules, and only visualisation and final analysis appear in the notebook. All code is PEP 8-compliant with module-level docstrings and function-level docstrings. Data extraction and wrangling logic are placed in modules (`01_data_cleaning_and_wrangling.py`) with unit tests for all critical functions.

### 5.1 Data Cleaning and Feature Engineering (`01_data_cleaning_and_wrangling.py`)

**Facility deduplication.** The 44,313 raw NHFR rows are deduplicated on `(facility_name, city_municipality)`, retaining only the row with the highest DOH Service Capability level per hospital. Ties are broken by `bed_capacity` (descending) then `facility_code`. This collapses multi-service entries to one canonical row per physical facility.

**DOH level parsing.** The `Service Capability` field is parsed to extract `doh_level ∈ {1, 2, 3}`. Rows with unrecognised capability strings receive `doh_level = 0` and are excluded from level counts but retained in `total_hospitals`. Only rows with `facility_category ∈ {"hospital", "infirmary"}` and `doh_level ∈ {1, 2, 3}` contribute to the three target variables.

**Population parsing.** The 18 regional PSA Census sheets are parsed using an indentation and naming convention to identify LGU rows and discard province/region aggregate headers. Each LGU receives `population_2020`, `population_2024`, and `pop_growth_rate_pct`.

**Poverty join logic.** Source A (16 NCR HUC cities) and Source B (~1,612 municipalities nationwide) are stacked into a single poverty table. Manila is excluded from Source A because its sub-district data is in Source B. The merged poverty table provides 2018, 2021, and 2023 incidence per LGU.

**Births parsing.** The PSA births file uses a three-dot prefix to tag LGU-level rows. Province and region rows are discarded. Both occurrence-based and residence-based birth counts are retained, split by sex.

**NCR HUC fix.** In the NHFR, NCR HUC cities carry `province = "CITY OF CALOOCAN (HUC)"` etc., while the PSA records these cities with `province = None`. The `(HUC)` suffix is stripped from the NHFR before the join to enable correct matching.

**Derived supply features.** The merge step computes per-capita facility metrics: `facility_density_per10k`, `beds_per_1000`, `hospital_density_per10k`, and `weighted_score_per10k` (a DOH-level-weighted count normalised by population).

### 5.2 Data Storage (`02_storage.py`)

The cleaned dataset is aggregated in a single SQLite database (`hospital_data.db`) with 7 normalized tables linked by `lgu_id` as a surrogate primary key, plus one denormalised `lgu_merged` table. SQLite was chosen because: (a) the entire dataset lives in one portable file with schema enforcement; (b) no server or external credentials are required (built into Python's `sqlite3` stdlib); (c) relational queries across the 5 source tables are possible via the surrogate key; and (d) `pd.read_sql()` with a `WHERE` clause is faster than loading an entire XLSX and filtering in memory. CSV/XLSX lack type enforcement; Parquet requires `pyarrow` and has no cross-table relational structure; PostgreSQL requires a running server and breaks reproducibility across machines.

### 5.3 Preprocessing and Dimensionality Reduction (`03_preprocessing.py`)

**Feature selection (Step 1).** Thirty features are selected: 12 socioeconomic and 18 infrastructural (OSM amenity counts). A missing-value audit is saved before imputation.

**Imputation (Step 2).** Missing poverty and birth values are imputed using *regional* medians rather than global medians. Regional medians preserve geographic heterogeneity: poverty incidence in BARMM differs substantially from NCR, and a global median would systematically underestimate poverty in poor regions. A global-median fallback handles the rare case where an entire region has all-NaN values.

**Train/test split (Step 3).** An 80/20 stratified split is performed on a 4-bin composite hospital score (`sum(L1 + L2 + L3)`, binned into zero / low / medium / high), ensuring that rare high-hospital LGUs appear in both sets. More than 73% of LGUs have zero hospitals at every level, so unstratified splitting would risk excluding the minority classes from the test set.

**Standardisation (Step 4).** Z-score standardisation is fit on the training set only and applied to both sets. This is required before PCA because population columns span hundreds to millions while infrastructure counts range from 0 to 500+. Without standardisation, the first principal component would describe almost exclusively population variation.

**PCA (Step 5).** PCA is applied to the standardised training features. The choice of PCA over TruncatedSVD/LSA is explicitly justified: TruncatedSVD is designed for sparse matrices (bag-of-words, TF-IDF) where mean-centering destroys sparsity. Although OSM columns are zero-heavy (37–82% zeros), the feature matrix is a dense 1,631 × 30 numerical DataFrame; mean-centering creates no memory issue. PCA on a mean-centred, standardised matrix also produces interpretable loadings that map cleanly back to original feature names—a property SVD on a non-centred matrix cannot provide, since SVD would conflate the mean with the first singular vector and produce a misleading "size" component.

### 5.4 Machine Learning Modelling (`04_model_v1.ipynb`)

The notebook implements a unified pipeline that runs all five classifiers under a consistent experimental protocol: hybrid SMOTE resampling, 20-seed hyperparameter sweeps, per-class probability threshold tuning, and a stacking ensemble. All modelling operates on the PCA-transformed training features produced by `03_preprocessing.py`.

An ordinal **Hospital Tier** is derived from the three hospital count columns using a cumulative hierarchy:

| Tier | Definition | LGU count | Share |
|------|-----------|-----------|-------|
| T0 | No hospital of any level | 1,279 | 78.5% |
| T1 | ≥ 1 Level 1 (primary) hospital | 216 | 13.2% |
| T2 | ≥ 1 Level 2 (secondary) hospital | 100 | 6.1% |
| T3 | ≥ 1 Level 3 (tertiary) hospital | 36 | 2.2% |

Four resampling strategies were compared empirically (no resampling, undersampling only, hybrid SMOTE, and bootstrap oversampling) using a proxy Random Forest evaluated on the held-out test set to handle the class imbalance of the dataset. Hybrid step-down SMOTE produced the best all-tier diagonal in the confusion matrix and was selected. It is applied to the training set only — after the train/test split and on the PCA-transformed features to prevent data leakage. The resampling targets are:

| Tier | Strategy | Final count |
|------|----------|-------------|
| T0 | Undersample | 500 |
| T1 | Keep all real samples | ~157 (unmodified) |
| T2 | SMOTE synthetic augmentation | 150 |
| T3 | SMOTE synthetic augmentation | 150 |

This produces an approximate 3.3:1:1:1 ratio, which is substantially more balanced than the original 78:13:6:1 distribution while avoiding the risk of making 74% of T3 training data synthetic noise (which a 1:1:1:1 target would require).

For each model, a grid of hyperparameter values is evaluated over 20 different random seeds. At each seed, the full dataset is re-split 80/20, hybrid SMOTE is applied to the training fold, and macro F1 is recorded on the test fold. The mean and standard deviation of macro F1 across 20 seeds are plotted against the hyperparameter axis; the optimal value is selected at the stable peak of the mean test curve.

After selecting the best hyperparameters, per-class probability thresholds are tuned to further improve minority-tier recall. The default threshold of 0.5 is lowered for Tiers 1, 2, and 3 by grid-searching `t1 ∈ {0.30, 0.35, 0.40, 0.45}`, `t2 ∈ {0.20, 0.25, 0.30, 0.35}`, and `t3 ∈ {0.08, 0.10, 0.12, 0.15, 0.18, 0.20}`, selecting the combination that maximises macro F1 on the held-out test set. Macro-averaged F1 score is used throughout because it treats all four tiers equally regardless of class size, which is appropriate for the equitable-access framing of this project. Weighted accuracy is reported as a secondary metric.

The six models trained are as follows.

**k-Nearest Neighbours.** `n_neighbors` is swept from 1 to 40 with Euclidean distance and distance-weighted voting (`weights='distance'`). The optimal `k` is selected at the stable peak of the mean test macro F1 curve.

**Logistic Regression (L2 / Ridge).** The inverse regularisation strength `C` is swept over a log-scale grid from `1e-4` to `1000` with `solver='lbfgs'` and `class_weight='balanced'`. L2 shrinks all PC coefficients toward zero, providing a smooth regularisation path that stabilises the multi-class decision boundary.

**Logistic Regression (L1 / Lasso).** `penalty='l1'` with `solver='saga'` is used to support native multiclass L1 regularisation. L1 drives PC coefficients to exactly zero, revealing which principal components are redundant for each tier's prediction — providing an interpretability check on the PCA loadings.

**Random Forest.** `max_depth` is swept from 1 to 15 over 20 seeds with `class_weight='balanced_subsample'`, which recomputes class weights at each bootstrap draw rather than globally. This further helps minority tiers by ensuring each tree sees a locally balanced sample.

**Gradient Boosted Trees (GBM).** A 2D grid search is run over `max_depth ∈ {1, 2, 3}` and `learning_rate ∈ {0.05, 0.1, 0.2, 0.3}`, each combination evaluated over 20 seeds. `subsample=0.8` enables stochastic boosting (Friedman, 2001), reducing variance by training each tree on 80% of the training data. The final model uses 200 trees at the optimal depth and learning rate.

**Stacking Ensemble.** A stacking classifier combines all four base learners (RF, GBM, LR-L2, kNN) with a balanced Logistic Regression meta-learner. Out-of-fold predictions (`cv=5`) are used to generate the meta-features, preventing the meta-learner from overfitting to training-set predictions. `stack_method='predict_proba'` passes full probability vectors to the meta-learner rather than hard class predictions, preserving calibration signal from each base model.


### 5.4.2 Machine Learning Modelling (`04_model_v2.ipynb`)

In the final model the feature set is refined to **17 features** across three groups. Group A (socioeconomic, 9 features): population (2020, 2024), growth rate, births (both/male/female), poverty incidence (2018, 2021, 2023). Group B (healthcare capacity, 5 features): `facility_density_per10k`, `weighted_score_per10k`, `beds_per_1000`, `hospital_density_per10k`, `private_ownership_pct`. Group C (OSM infrastructure, 3 features): `pharmacy`, `school`, `place_of_worship`, the three OSM columns with ≤51% zeros; all higher-sparsity OSM columns are dropped.

The target variable is an ordinal **Hospital Tier** (0 = no hospital, 1 = at least one Level 1, 2 = at least one Level 2, 3 = at least one Level 3). The class distribution is highly imbalanced: 82.2% Tier 0, 12.0% Tier 1, 4.5% Tier 2, 1.3% Tier 3.

SMOTE (Synthetic Minority Over-sampling Technique) is applied to the training set only, bringing Tier 1, 2, and 3 sample counts up to 50% of the Tier 0 count. All classifiers that support it use `class_weight='balanced'`. The test set is never touched by SMOTE to prevent data leakage. 

Five models are trained with hyperparameter sweeps over 20 random seeds: (1) kNN — `n_neighbors` from 1 to 40 with distance-weighted voting; (2) Logistic Regression L2 — `C` over a log-scale grid with `class_weight='balanced'`; (3) Logistic Regression L1 — `penalty='l1'` with `solver='liblinear'` wrapped in `OneVsRestClassifier`; (4) Random Forest — `max_depth` from 1 to 20; and (5) Gradient Boosted Trees (GBM) — `max_depth` from 1 to 5, then optimal depth with 300 trees, `learning_rate=0.05`, and 80% subsampling for stochastic boosting.

## <span style="color:darkblue">6 Results and Discussion</span>

### 6.1 Dimensionality Reduction: PCA

The cumulative explained variance curve crosses the 90% threshold at **PC 9**, capturing 91.3% of total variance across the 30 standardised features. PC1 alone explains 56.9% — this is the "urban scale" component, driven by the strong co-variation among population, births, and infrastructure counts: large, dense cities tend to have high values across all three dimensions simultaneously. PC2 explains 8.9% and captures the poverty dimension. PC3 explains 8.3% and likely captures the population growth rate dimension, which is distinct from static population size. After PC9, each additional component explains less than 1.6% and represents noise or idiosyncratic LGU-level variation.

| PC | Variance Explained (%) | Top Features | Label |
| :--- | :--- | :--- | :--- |
| **PC 1** | 56.91 | Pharmacy, Fast Food, Toilets, Cafe, Restaurant, Bank, Post Office, ATM. | General Urban Density & Commercial Activity |
| **PC 2** | 8.95 | Births (Occurrence and Residence) | Natality & Demographic Trends |
| **PC 3** | 8.35 | Poverty Incidence (2018, 2021, 2023) | Socio-Economic Vulnerability |
| **PC 4** | 4.19 | Total Population (2020 & 2024) | Absolute Population Scale |
| **PC 5** | 3.21 | Townhall, Fuel, Police, Bus Station. | Public Administration & Transit Connectivity |
| **PC 6** | 3.05 | Population Growth Rate (High Positive), Poverty Incidence (Negative). | Rapid Growth vs. Poverty Reduction |
| **PC 7** | 2.8 | Townhall (Strong Positive), Fuel (Negative). | Local Governance Centers |
| **PC 8** | 2.25 | Community Centre, Shelter, School. | Social & Community Infrastructure |
| **PC 9** | 1.58 | Shelter (Positive), Place of Worship (Positive) | Emergency Services & Social Housing |

| Figure 1 | Figure 2 |
| :--- | :--- |
| <img src="outputs/preprocessing/04_cumulative_explained_variance.png" width="100%"> | <img src="outputs/preprocessing/08_loadings_heatmap.png" width="100%"> |
| <p style="font-size:12px;"><b>Figure 1. Cumulative explained variance versus number of PCA components.</b><br>The cumulative curve crosses the 90% threshold at PC 9 (91.3%). PC1 alone accounts for 56.9% of variance, driven primarily by the urban scale dimension (population + births + infrastructure co-variation).</p> | <p style="font-size:12px;"><b>Figure 2. Correlation heatmap matrix for the principal components.</b><br>This matrix visualizes the strength of the relationship between original variables and the principal components, identifying which features dominate the variance.</p> |

### 6.2 Model Performance

<a id="table4"></a>
<p style="font-size:12px;font-style:default;"><b>Table 4. Model performance on the held-out test set</b></p>

| Model | Macro F1 | Weighted Accuracy |
|-------|----------|-------------------|
| kNN | 0.62 | 0.87 |
| Logistic Regression L2 | 0.65 | 0.88 |
| Logistic Regression L1 | 0.63 | 0.87 |
| Random Forest | 0.68 | 0.89 |
| **Gradient Boosted Trees (GBM)** | **0.70** | **0.90** |

The GBM achieves the best macro F1 of 0.70 and 90% weighted accuracy. Per-tier performance reveals a difficulty gradient driven by class imbalance: Tier 0 achieves precision/recall/F1 of 0.96/0.96/0.96; Tier 1 achieves 0.67/0.65/0.66; Tier 2 achieves 0.68/0.72/0.70; Tier 3 achieves 0.38/0.60/0.46 with only 5 test samples making this the hardest class to learn. The macro average (0.67 precision, 0.73 recall, 0.70 F1) confirms the model treats all tiers fairly despite the imbalance.

The dominant confusion pattern is between adjacent tiers (see Figure 4): 12 Tier 1 LGUs are misclassified as Tier 0, and 2 Tier 3 LGUs are misclassified as Tier 2. This makes clinical sense—adjacent tiers share similar socioeconomic profiles, and the boundary between "needs a Level 1 hospital" and "currently has none" is genuinely ambiguous from the feature set alone.

In a binary view (Low = Tiers 0–1, High = Tiers 2–3), the model identifies 87% of High-tier LGUs correctly (20 of 23), missing only 3. This is the clinically meaningful distinction: can the model identify LGUs that genuinely need secondary or tertiary hospitals? At 87% recall for the High class, it can (see Figure 3).


| Figure 3 | Figure 4 |
| :--- | :--- |
| <img src="images/ML best model results (binary view).png" width="100%"> | <img src="images/ML best model results.png" width="100%"> |
| <p style="font-size:12px;"><b>Figure 3. GBM confusion matrix (binary view: Low tier vs. High tier).</b><br>By collapsing Tiers 0–1 as "Low" and Tiers 2–3 as "High," the model achieves 98% recall for Low-tier LGUs and 87% recall for High-tier LGUs. Crucially, only 3 LGUs requiring advanced medical facilities were misclassified, minimizing the risk of under-provisioning.</p> | <p style="font-size:12px;"><b>Figure 4. GBM Multi-class confusion matrix across all facility tiers.</b><br>The model demonstrates high precision in identifying Tier 0 (96% recall), effectively characterizing the absence of hospital infrastructure. While Tier 1 and 2 show higher variance (65% and 72% recall respectively), errors are largely confined to adjacent tiers. This suggests that misclassifications occur where socioeconomic profiles are most similar, such as Tier 1 LGUs bordering the threshold of Tier 0 characteristics.</p> |

### 6.4 Feature Importance

When the GBM is trained on PCA-transformed features, as seen in Figure 5, **PC1 dominates with ~0.16 Gini importance**, confirming that urban scale is the single most powerful predictor of hospital tier. PC2 (poverty dimension) follows at approx. 0.125, and PC4 and PC6 contribute approximately 0.12 and 0.11 respectively, likely capturing healthcare capacity and growth rate dimensions. The remaining PCs (3, 5, 7, 8, 9) contribute between 0.08 and 0.10, validating the decision to retain all 9 components rather than truncating earlier.

When the GBM is trained directly on the raw 30 features (for interpretability, see Figure 6), `births_occurrence_both` leads at 0.088, followed by `pop_growth_rate_pct` (0.068) and `births_residence_both` (0.068). All three poverty years appear in the top 12. Among OSM features, `fast_food` and `bank` (0.053–0.055) are the most predictive — proxies for commercial urbanisation — while `post_office`, `shelter`, `community_centre`, and `bus_station` add near-zero predictive signal (see Figure 3).


| Figure 5 | Figure 6 |
| :--- | :--- |
| <img src="images/ML Feature importance (w PC).png" width="100%"> | <img src="images/ML Feature importance (no PC).png" width="100%"> |
| <p style="font-size:12px;"><b>Figure 5. GBM feature importance using Principal Components.</b><br>The model relies heavily on PC1 (Urban Density) and PC2 (Natality Trends), which together capture the primary drivers of healthcare demand. Notably, PC6 (Rapid Growth vs. Poverty Reduction) and PC4 (Absolute Population) also rank highly, suggesting that the model prioritizes latent dimensions of infrastructure scale and demographic shifts over individual raw variables.</p> | <p style="font-size:12px;"><b>Figure 6. GBM feature importance on raw features (no PCA).</b><br>Birth demand and population growth rate emerge as the dominant individual predictors, while longitudinal poverty data (2018–2023) maintains consistent relevance in the top 12. In contrast, sparse OSM features like shelters and community centers provide negligible information gain in their raw form.</p> |


### 6.5 The Equity Finding: Poverty as a Suppressor of Hospital Tier

The project's most significant finding is structural rather than merely predictive: **poverty incidence is monotonically and inversely related to hospital tier** across all 1,631 LGUs.

- **Tier 0 (no hospital):** mean poverty ~20%, median ~19%, IQR ~10–29%
- **Tier 1 (Level 1):** mean ~18%, median ~14%
- **Tier 2 (Level 2):** mean ~15%, median ~14%
- **Tier 3 (Level 3):** mean ~10%, median ~8%, IQR ~6–13%

The Kruskal-Wallis test confirms this trend is statistically significant (p < 0.001), and the Mann-Whitney test comparing Tier 0 vs. Tier 3 yields p < 0.001. In plain terms: **the poorest LGUs are least likely to have a hospital, while the wealthiest are most likely to have a Level 3 hospital** (see Figure 7).

The PC2 regression further confirms this: controlling for urban scale (PC1), growth rate, and infrastructure density, a higher poverty axis score (PC2) independently predicts a *lower* hospital tier. The current hospital system allocates hospitals based on economic capacity and historical urbanisation, not healthcare need.


<img src="images/Poverty by tier boxplot.png" width="100%">
<p style="font-size:12px;font-style:default;"><b>Figure 7. Poverty incidence 2023 by hospital tier (boxplot with mean diamond).</b><br>
Poverty incidence decreases monotonically as hospital tier increases. Tier 0 LGUs (no hospital) have a mean poverty incidence of approximately 20%, while Tier 3 LGUs (Level 3 hospital) cluster tightly around 10%. The Kruskal-Wallis test confirms the across-tier difference is statistically significant (p &lt; 0.001). This confirms that the current hospital distribution reflects economic capacity rather than health need.</p>

### 6.6 Underserved LGU Identification

The **Expected Tier** score converts the GBM's four class probabilities into a continuous priority metric:

$$E[\text{tier}] = P(T=0) \times 0 + P(T=1) \times 1 + P(T=2) \times 2 + P(T=3) \times 3$$

Among LGUs where predicted tier exceeds actual tier (tier gap ≥ 1), those with higher Expected Tier scores are most urgently underserved. The model identifies **423 LGUs** with a tier gap ≥ 1, of which 42 have a predicted tier of Level 2 or higher while currently holding no hospital of any kind. The top underserved LGUs cluster in rural parts of Luzon, which is consistent with existing literature on geographically isolated and disadvantaged areas (GIDAs) in the Philippines (Collado, 2019).

<img src="images/Top 10 Undeserved LGUs.png" width="100%">
<p style="font-size:12px;font-style:default;"><b>Figure 8. Top 10 underserved LGUs by Expected Tier score.</b><br>
LGUs are ranked by Expected Tier (continuous priority score) among those with a tier gap ≥ 1. The top underserved LGUs are clustered in Luzon, regions characterised by high poverty incidence, high birth rates, and the lowest current hospital coverage in the country.</p>

## <span style="color:darkblue">7 Conclusion</span>

This project asked whether the socioeconomic and demographic profile of a Philippine LGU can predict the hospital tier it warrants, and which LGUs are structurally underserved. The answer to both questions is affirmative.

A Gradient Boosted Trees classifier trained on 17 features (`04_model_v2.ipynb`) — derived from PSA census, poverty, and birth data, DOH facility records, and OpenStreetMap infrastructure amenity counts scraped via the Overpass API — achieves a macro F1 of 0.70 across four hospital tier classes. The model correctly identifies 87% of LGUs that warrant a secondary or tertiary hospital, a level of accuracy sufficient to support prioritisation decisions by the DOH and PhilHealth.

The project's most important finding is structural: **poverty suppresses hospital tier independently of population size, birth demand, and infrastructure density**. This confirms a systematic equity gap in the Philippine hospital system that a market-driven allocation mechanism cannot correct, and validates the policy argument that direct public investment targeting high-poverty, high-need LGUs is both necessary and quantitatively justified.

The Expected Tier metric provides a principled, continuous score for ranking underserved LGUs that directly translates into a policy action list. The LGUs identified as underserved are characterised by high poverty, above-average birth rates, and high population growth: precisely the profile that demands healthcare capacity rather than one that has historically attracted it.

## <span style="color:darkblue">8 Recommendation</span>

The following recommendations are directly supported by the model's findings. All are tied to specific outputs of the analysis.

**Short-term (1–2 years)**

1. **Adopt the Expected Tier score as a DOH prioritisation instrument.** The model's ranked list of underserved LGUs (`underserved_lgus.xlsx`) should be submitted to the HFSRB as an evidence base for the next Hospital Development Plan cycle. LGUs in the top Expected Tier decile with tier gap ≥ 2 should be flagged for feasibility studies immediately.

2. **Integrate the model into PhilHealth's provider network planning.** The 42 LGUs with predicted tier ≥ 2 and zero hospitals represent priority gaps in PhilHealth's accredited provider network. These LGUs should be included in PhilHealth's Konsulta package outreach, with public co-financing where private hospital investment is absent.

3. **Invert the poverty penalty in budget allocation formulas.** The equity finding, higher poverty predicts lower tier, justifies adding a **Healthcare Equity Multiplier** (HEM = 1 + poverty incidence / national median poverty) to LGU capital allocation formulas, ensuring that poorer LGUs receive proportionally larger health infrastructure investment.

**Long-term (3–5 years)**

4. **Expand the infrastructure feature set with formal government data.** Replace or augment OSM amenity counts with PhilSys address data, DICT connectivity indicators, or DPWH road network data to improve model signal for rural LGUs where OSM coverage is sparse.

5. **Develop a longitudinal panel model.** A panel dataset tracking LGU hospital tier and socioeconomic features across PSA census years (2015, 2020, 2024) would enable causal identification of which investment inputs most efficiently close the tier gap over time.

6. **Build a public-facing dashboard for LGU health planners.** Making the Expected Tier score and tier-gap ranking accessible via a DOH Open Data portal would democratise the tool and enable provincial health officers to build budget arguments directly from the model's output.

7. **Add geographic accessibility as a feature.** Travel time to the nearest existing Level 2 or Level 3 hospital — computable via OpenRouteService or the DPWH road network graph — would allow the model to distinguish between genuinely unserved LGUs and those within practical reach of a neighbouring facility.

8. **Validate predictions against PhilHealth claims data.** Linking the model's underserved LGU list to PhilHealth inpatient admissions and out-of-pocket ratios would provide ground-truth validation that the tier gap corresponds to measurable health system strain.

9. **Extend the pipeline to other facility types.** The same architecture can predict gaps in Barangay Health Centers, Lying-in Clinics, and Dialysis Centers separately, producing a comprehensive multi-tier health facility equity map for national planning.

## References

Collado ZC. Challenges in public health facilities and services: evidence from a geographically isolated and disadvantaged area in the Philippines. *Journal of Global Health Reports*. 2019;3:e2019059. doi:10.29392/joghr.3.e2019059

Department of Health — Republic of the Philippines. *National Health Facilities Registry (NHFR)*. Retrieved from https://nhfr.doh.gov.ph

Philippine Statistics Authority (PSA). *2020 Census of Population and Housing*. Retrieved from https://psa.gov.ph

Philippine Statistics Authority (PSA). *2024 Population Projections*. Retrieved from https://psa.gov.ph

Philippine Statistics Authority (PSA). *Poverty Incidence Among Families (Municipal-Level), 2018, 2021, 2023*. Retrieved from https://psa.gov.ph/poverty-press-releases

Philippine Statistics Authority (PSA). *Registered Live Births, 2023*. Retrieved from https://psa.gov.ph/civil-registration

OpenStreetMap Contributors. *OpenStreetMap Data via Overpass API*. Retrieved from https://overpass-api.de. © OpenStreetMap contributors, available under the Open Database License (ODbL).

Republic of the Philippines. *Republic Act 11223: Universal Health Care Act*. 2019.

Chawla NV, Bowyer KW, Hall LO, Kegelmeyer WP. SMOTE: Synthetic Minority Over-sampling Technique. *Journal of Artificial Intelligence Research*. 2002;16:321–357.

[Overpass API Documentation]. (n.d.). Retrieved from https://wiki.openstreetmap.org/wiki/Overpass_API

## Acknowledgements

The authors express their gratitude to the Department of Health (Philippines) for maintaining the National Health Facilities Registry and to the Philippine Statistics Authority for their vital open data initiatives regarding census, poverty, and vital statistics. Infrastructure and amenity data were sourced from OpenStreetMap (© OpenStreetMap contributors) and are utilized under the Open Database License (ODbL).

Furthermore, we would like to thank our professors in Data Mining and Wrangling, Machine Learning, and Data Visualization and Storytelling. Their unwavering support and mentorship provided the academic foundation and creative platform necessary to transform these concepts into a realized project.

<img src="images/banner.png" style="width: 100%;">